# MiniRocket + ridge (seed 0)

1. Check the model on the train/test split from `train_test_split_process_A.ipynb` (process A).
2. Train on all calibration data (A and B) and write a table in the leaderboard format.

Needs `aeon`: `pip install aeon`. The data folder is only read, never written.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from aeon.transformations.collection.convolution_based import MiniRocket
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
LENGTH = 500  # every batch becomes 500 points per sensor

# first data folder that exists (Renku or a local copy)
candidates = ["../../input-data/Project 2 - BRPMM", "../input-data/Project 2 - BRPMM",
              "/home/renku/work/input-data/Project 2 - BRPMM", "../../Project 2 - BRPMM"]
DATA_DIR = next(Path(c) for c in candidates if Path(c).is_dir())
print(DATA_DIR.resolve())

In [ ]:
def load_batches(csv_path):
    """One array (rows x sensors) per batch."""
    df = pd.read_csv(csv_path)
    sensors = [c for c in df.columns if c != "batch_id"]
    return {b: g[sensors].to_numpy(float) for b, g in df.groupby("batch_id")}


def fixed_length(a):
    """Cut the batch into LENGTH equal slices and average each one -> (sensors x LENGTH)."""
    n = len(a)
    if n < LENGTH:  # very short batch: stretch it instead
        old, new = np.linspace(0, 1, n), np.linspace(0, 1, LENGTH)
        return np.stack([np.interp(new, old, a[:, j]) for j in range(a.shape[1])])
    slice_id = np.arange(n) * LENGTH // n
    sums = np.zeros((LENGTH, a.shape[1]))
    np.add.at(sums, slice_id, a)
    return (sums / np.bincount(slice_id)[:, None]).T


def to_array(batches, ids):
    return np.stack([fixed_length(batches[b]) for b in ids])  # (batches, sensors, LENGTH)


class Model:
    """MiniRocket turns each batch into ~10,000 numbers; a ridge classifier decides."""

    def fit(self, X, y, alpha=None):
        self.mean = X.mean(axis=(0, 2), keepdims=True)  # put every sensor on the same scale
        self.std = X.std(axis=(0, 2), keepdims=True) + 1e-12
        self.rocket = MiniRocket(random_state=SEED).fit(self._scale(X))
        features = self.rocket.transform(self._scale(X))
        self.scaler = StandardScaler().fit(features)
        if alpha is None:  # pick alpha (ridge strength) by 4-fold CV on F1 of "rejected"
            search = GridSearchCV(RidgeClassifier(class_weight="balanced"), {"alpha": np.logspace(-1, 5, 7)},
                                  scoring="f1", cv=StratifiedKFold(4, shuffle=True, random_state=SEED))
            alpha = search.fit(self.scaler.transform(features), y).best_params_["alpha"]
        print("alpha:", alpha)
        self.ridge = RidgeClassifier(alpha=alpha, class_weight="balanced").fit(self.scaler.transform(features), y)
        return self

    def predict(self, X):
        features = self.rocket.transform(self._scale(X))
        return self.ridge.predict(self.scaler.transform(features))  # 1 = rejected

    def _scale(self, X):
        return (X - self.mean) / self.std

## 1. Check on the colleague's split (process A)

Same split as `train_test_split_process_A.ipynb`: 30% of the batches for testing, stratified by label, `random_state=42`.

In [ ]:
folder = DATA_DIR / "process A" / "calibration data"
batches = load_batches(folder / "calibration_data.csv")
labels = pd.read_csv(folder / "calibration_labels.csv")
is_rejected = labels.set_index("batch_id").label.eq("rejected").astype(int)

train_ids, test_ids = train_test_split(labels.batch_id, test_size=0.30, random_state=42, stratify=labels.label)

model = Model().fit(to_array(batches, train_ids), is_rejected[train_ids].to_numpy())
pred = model.predict(to_array(batches, test_ids))
true = is_rejected[test_ids].to_numpy()

print(f"F1 rejected {f1_score(true, pred):.3f} | precision {precision_score(true, pred):.3f} | "
      f"recall {recall_score(true, pred):.3f} | accuracy {accuracy_score(true, pred):.3f}")
print(f"rejected caught: {((pred == 1) & (true == 1)).sum()} of {true.sum()}, "
      f"false alarms: {((pred == 1) & (true == 0)).sum()} of {(true == 0).sum()}")

## 2. Train on all calibration data and predict the test data

One model per process. Alpha is fixed to the value `src/compare_models.py` picked with its more careful
cross-validation, so this gives the same table. The table goes to `outputs/` (not sent anywhere).

In [ ]:
ALPHA = {"A": 100.0, "B": 0.1}  # chosen by src/compare_models.py

predictions = []
for p in ["A", "B"]:
    folder = DATA_DIR / f"process {p}"
    batches = load_batches(folder / "calibration data" / "calibration_data.csv")
    labels = pd.read_csv(folder / "calibration data" / "calibration_labels.csv")
    model = Model().fit(to_array(batches, labels.batch_id), labels.label.eq("rejected").astype(int).to_numpy(), ALPHA[p])

    test = load_batches(folder / "test data" / "test_data.csv")
    ids = sorted(test)
    pred = model.predict(to_array(test, ids))
    predictions.append(pd.DataFrame({"batch_id": ids, "label": np.where(pred == 1, "rejected", "accepted")}))
    print(f"process {p}: {pred.sum()} of {len(pred)} test batches predicted rejected")

# same rows and order as the template
template = pd.read_csv(DATA_DIR / "BRPMM_template.csv")
template.columns = template.columns.str.strip()
template["batch_id"] = template.batch_id.str.strip()
submission = template[["batch_id", "project"]].merge(pd.concat(predictions), on="batch_id", how="left")
assert len(submission) == len(template) and submission.label.notna().all()

Path("../outputs").mkdir(exist_ok=True)
submission.to_csv("../outputs/submission_minirocket_seed0.csv", index=False)
submission.groupby(["project", "label"]).size()